# SAM Generation/Comparison
This notebook generates the original and augmented images masks and compares them with three different metrics:
- Mean IoU
- Mean Boundary IoU
- Mean Mask Count Diff

Note that if you cloned this repo the masks have already been generated at `/training_data/masks.zip`, be sure to modify the folder paths at the configuration cell.

NOTE: We used Google Colab to run this notebook as it requires a lot of GPU power depending on which SAM checkpoint you use, beware!

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Dependencies

In [ ]:
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q opencv-python-headless pandas matplotlib

# edit this if u would rather use a different model (e.g. sam_vit_l.pth)
import os
if not os.path.exists('sam_vit_b.pth'):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O sam_vit_b.pth

## 3. Configuration
Contains the folder paths u need to edit

In [ ]:
from pathlib import Path

# ── Edit this to match your Google Drive structure ────────────────────────────
BASE_DIR  = Path('/content/drive/MyDrive/SAM_Project/training_data')   # folder containing all image folders
MASK_DIR  = Path('/content/drive/MyDrive/SAM_Project/masks')  # masks will be saved here
# ─────────────────────────────────────────────────────────────────────────────

CATEGORIES   = ['Dog', 'Mobile_phone', 'Train']
AUGMENTATIONS = ['Gaussian', 'Motion', 'Compression', 'All']

MASK_DIR.mkdir(parents=True, exist_ok=True)
print(f'Base: {BASE_DIR}')
print(f'Masks: {MASK_DIR}')

## 4. SAM Model

In [ ]:
import torch
import numpy as np
import cv2
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

device = 'cuda' if torch.cuda.is_available() else 'cpu'

sam = sam_model_registry['vit_b'](checkpoint='sam_vit_b.pth')
sam.to(device)

mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,        
    pred_iou_thresh=0.88,
    stability_score_thresh=0.95,
)

## 5. Generate & Save Masks
Runs SAM over the given images, will skip already processed images (checks based on file names).

In [ ]:
def generate_and_save_masks(image_dir: Path, mask_out_dir: Path):
    mask_out_dir.mkdir(parents=True, exist_ok=True)
    image_files = sorted(image_dir.glob('*.jpg'))

    for img_path in image_files:
        out_path = mask_out_dir / f'{img_path.stem}.npy'

        # Skip if already processed
        if out_path.exists():
            print(f'--- skip {img_path.name} (already done)')
            continue

        image = cv2.imread(str(img_path))
        if image is None:
            print(f'[!] Could not read {img_path.name}')
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        masks = mask_generator.generate(image)

        mask_array = np.stack([m['segmentation'] for m in masks])
        np.save(str(out_path), mask_array)

for category in CATEGORIES:
    folders = [category] + [f'{category}_{aug}' for aug in AUGMENTATIONS]
    for folder_name in folders:
        img_dir  = BASE_DIR / folder_name
        mask_dir = MASK_DIR / folder_name
        if not img_dir.is_dir():
            print(f'[!] Folder not found, skipping: {img_dir}')
            continue
        generate_and_save_masks(img_dir, mask_dir)

## 6. Metric Functions

In [ ]:
def compute_iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    intersection = np.logical_and(mask_a, mask_b).sum() # Count where both masks overlap
    union        = np.logical_or(mask_a, mask_b).sum() # Count where either mask exists
    return float(intersection / union) if union > 0 else 0.0


def get_boundary(mask: np.ndarray, dilation: int = 2) -> np.ndarray:
    mask_uint8 = mask.astype(np.uint8) # Convert boolean mask to uint8 for OpenCV
    kernel     = np.ones((2 * dilation + 1, 2 * dilation + 1), np.uint8) # Create a square kernel for dilation
    eroded     = cv2.erode(mask_uint8, kernel) # Erode to find inner area
    return mask_uint8 - eroded


def compute_boundary_iou(mask_a: np.ndarray, mask_b: np.ndarray, dilation: int = 2) -> float:
    b_a = get_boundary(mask_a, dilation).astype(bool)
    b_b = get_boundary(mask_b, dilation).astype(bool)
    intersection = np.logical_and(b_a, b_b).sum() # Count where both boundaries overlap
    union        = np.logical_or(b_a, b_b).sum() # Count where either boundary exists
    return float(intersection / union) if union > 0 else 0.0

# AI Generated match_and_score function that helps match masks between original and augmented sets and computes mean IoU and boundary IoU scores.
def match_and_score(orig_masks: np.ndarray, aug_masks: np.ndarray) -> dict:
    iou_scores  = []
    biou_scores = []
    used        = set()

    for o_mask in orig_masks:
        best_iou, best_biou, best_idx = 0.0, 0.0, -1

        for i, a_mask in enumerate(aug_masks):
            if i in used:
                continue
            iou = compute_iou(o_mask, a_mask)
            if iou > best_iou:
                best_iou  = iou
                best_biou = compute_boundary_iou(o_mask, a_mask)
                best_idx  = i

        if best_idx >= 0:
            used.add(best_idx)

        iou_scores.append(best_iou)    # 0 if no match found
        biou_scores.append(best_biou)

    return {
        'mean_iou':        np.mean(iou_scores),
        'mean_boundary_iou': np.mean(biou_scores),
        'orig_mask_count': len(orig_masks),
        'aug_mask_count':  len(aug_masks),
        'mask_count_diff': len(aug_masks) - len(orig_masks),
    }

print('Metric functions ready.')

## 7. Run Comparisons

In [ ]:
import pandas as pd

results = []

for category in CATEGORIES:
    orig_mask_dir = MASK_DIR / category
    orig_files    = sorted(orig_mask_dir.glob('*.npy'))

    for aug in AUGMENTATIONS:
        aug_mask_dir = MASK_DIR / f'{category}_{aug}'

        for orig_path in orig_files:
            aug_stem = f'{orig_path.stem}_{aug}'
            aug_path = aug_mask_dir / f'{aug_stem}.npy'

            if not aug_path.exists():
                print(f'Missing: {aug_path.name}')
                continue

            orig_masks = np.load(str(orig_path))
            aug_masks  = np.load(str(aug_path))

            scores = match_and_score(orig_masks, aug_masks)
            results.append({
                'category':          category,
                'augmentation':      aug,
                'image':             orig_path.stem,
                **scores
            })

df = pd.DataFrame(results)
df.head(10)

## 8. Summary Table

In [ ]:
summary = df.groupby(['category', 'augmentation']).agg(
    mean_iou            = ('mean_iou',            'mean'),
    mean_boundary_iou   = ('mean_boundary_iou',   'mean'),
    mean_mask_count_diff= ('mask_count_diff',      'mean'),
).round(4)

print(summary.to_string())

## 9. Export Results to CSV

In [ ]:
results_path = Path('/content/drive/MyDrive/SAM_Project/results.csv')
df.to_csv(results_path, index=False)
print(f'Results saved to {results_path}')

## 10. Visualise Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
metrics = ['mean_iou', 'mean_boundary_iou']
titles  = ['Mean IoU', 'Mean Boundary IoU']

for ax, metric, title in zip(axes, metrics, titles):
    pivot = df.groupby(['augmentation', 'category'])[metric].mean().unstack()
    pivot.plot(kind='bar', ax=ax, rot=0)
    ax.set_title(title)
    ax.set_xlabel('Augmentation')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1)
    ax.legend(title='Category')

plt.tight_layout()
plot_path = '/content/drive/MyDrive/SAM_Project/results_plot.png'
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Plot saved to {plot_path}')